# Hybrid Approach: Best CNN + CV Refinement

**Цель:** Комбинировать лучшую CNN-модель с CV-уточнением для повышения точности детекции.

**Подход:**
1. CNN (YOLO или RCNN) генерирует кандидаты
2. CV refiner фильтрует по aspect ratio, edge density
3. Сравнение: CNN vs Hybrid


## 0. Colab Setup

⚠️ **Запустить только один раз!** Клонирует репозиторий (sparse checkout) и устанавливает зависимости.

In [ ]:
%%bash
cd /content
rm -rf aie-group-2-sapar
git init aie-group-2-sapar
cd aie-group-2-sapar
git sparse-checkout set project
git remote add origin https://github.com/Sapar-hub/aie-group-2-sapar.git
git pull origin main
cd project
pip install -q ultralytics opencv-python-headless pyyaml

In [ ]:
%cd /content/aie-group-2-sapar/project

In [ ]:
!nvidia-smi

## 1. Imports & Data Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_DIR = Path.cwd()
sys.path.insert(0, str(PROJECT_DIR / "src"))

import numpy as np
import matplotlib.pyplot as plt
import cv2
from ultralytics import YOLO

from evaluation.metrics import DetectionResult, bbox_iou, yolo_to_pixel, compute_metrics, print_metrics
from data.loader import load_image_and_labels
from hybrid.refiner import HybridRefiner

DATA_DIR = PROJECT_DIR / "data"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)
(ARTIFACTS_DIR / "metrics").mkdir(exist_ok=True)
(ARTIFACTS_DIR / "figures").mkdir(exist_ok=True)

IMAGE_TEST_DIR = DATA_DIR / "images" / "test"
LABEL_TEST_DIR = DATA_DIR / "labels" / "test"

print(f"Working dir: {PROJECT_DIR}")
print(f"Test images: {len(list(IMAGE_TEST_DIR.glob('*.png')))}")

## 2. Load Best Model

Загружаем лучшую модель (YOLO или RCNN) из предыдущих экспериментов.

In [ ]:
yolo_weights = ARTIFACTS_DIR / "yolo" / "exp01" / "weights" / "best.pt"

if yolo_weights.exists():
    model = YOLO(str(yolo_weights))
    model_type = "yolo"
    print(f"Loaded YOLO weights from {yolo_weights}")
else:
    print("YOLO weights not found, using pretrained")
    model = YOLO("yolov8n.pt")
    model_type = "yolo_pretrained"

refiner = HybridRefiner(expected_ar=3.55, ar_refine_tol=0.25, iou_threshold=0.3)

## 3. Evaluate CNN-only vs Hybrid

In [ ]:
all_images = sorted(IMAGE_TEST_DIR.glob("*.png")) + sorted(IMAGE_TEST_DIR.glob("*.jpg"))

cnn_results = []
hybrid_results = []

for img_path in all_images:
    img, labels = load_image_and_labels(img_path, LABEL_TEST_DIR)
    h, w = img.shape[:2]
    gt_bbox = yolo_to_pixel(tuple(labels[0]), w, h) if len(labels) > 0 else None
    
    preds = model(img, conf=0.1, verbose=False)
    if preds[0].boxes and len(preds[0].boxes) > 0:
        box = preds[0].boxes[0]
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        cnn_bbox = (x1, y1, x2 - x1, y2 - y1)
        all_bboxes = [(x1, y1, x2-x1, y2-y1) for box in preds[0].boxes]
    else:
        cnn_bbox = None
        all_bboxes = []
    
    hybrid_bbox = refiner.refine(all_bboxes, img) if all_bboxes else None
    
    cnn_iou = bbox_iou(cnn_bbox, gt_bbox) if cnn_bbox and gt_bbox else 0.0
    hybrid_iou = bbox_iou(hybrid_bbox, gt_bbox) if hybrid_bbox and gt_bbox else 0.0
    
    cnn_results.append(DetectionResult(
        image_name=img_path.name, gt_bbox=gt_bbox, pred_bbox=cnn_bbox,
        iou=cnn_iou, found=cnn_bbox is not None
    ))
    hybrid_results.append(DetectionResult(
        image_name=img_path.name, gt_bbox=gt_bbox, pred_bbox=hybrid_bbox,
        iou=hybrid_iou, found=hybrid_bbox is not None
    ))

cnn_metrics = compute_metrics(cnn_results, iou_threshold=0.5)
hybrid_metrics = compute_metrics(hybrid_results, iou_threshold=0.5)

print("=== CNN-only ===")
print_metrics(cnn_metrics, prefix="CNN  ")
print("\n=== Hybrid (CNN + CV Refine) ===")
print_metrics(hybrid_metrics, prefix="Hyb  ")

## 4. Comparison Table

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "Metric": ["IoU mean", "IoU std", "Precision", "Recall", "F1", "Detection rate"],
    "CNN": [
        round(cnn_metrics['iou_mean'], 3),
        round(cnn_metrics['iou_std'], 3),
        round(cnn_metrics['precision'], 3),
        round(cnn_metrics['recall'], 3),
        round(cnn_metrics['f1'], 3),
        round(cnn_metrics['detection_rate'], 3),
    ],
    "Hybrid": [
        round(hybrid_metrics['iou_mean'], 3),
        round(hybrid_metrics['iou_std'], 3),
        round(hybrid_metrics['precision'], 3),
        round(hybrid_metrics['recall'], 3),
        round(hybrid_metrics['f1'], 3),
        round(hybrid_metrics['detection_rate'], 3),
    ],
})
print(comparison.to_string(index=False))

## 5. Visualization

In [ ]:
sorted_hybrid = sorted(hybrid_results, key=lambda r: r.iou)
worst = sorted_hybrid[0]
best = sorted_hybrid[-1]

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
for ax, result, title_prefix in zip(axes, [worst, best], ["Worst", "Best"]):
    img_path = [p for p in all_images if p.name == result.image_name][0]
    img, _ = load_image_and_labels(img_path, LABEL_TEST_DIR)
    vis = img.copy()
    if result.gt_bbox:
        x, y, bw, bh = result.gt_bbox
        cv2.rectangle(vis, (x, y), (x+bw, y+bh), (0, 255, 0), 3)
    if result.pred_bbox:
        x, y, bw, bh = result.pred_bbox
        cv2.rectangle(vis, (x, y), (x+bw, y+bh), (0, 0, 255), 2)
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{title_prefix} IoU={result.iou:.3f} (Hybrid)")
    ax.axis("off")

plt.suptitle("Green=GT, Red=Pred (Hybrid)")
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "figures" / "hybrid_best_worst.png", dpi=150)
plt.show()

## 6. Conclusions

In [ ]:
summary = {
    "model": f"Hybrid ({model_type} + CV refiner)",
    "iou_mean_cnn": round(cnn_metrics['iou_mean'], 3),
    "iou_mean_hybrid": round(hybrid_metrics['iou_mean'], 3),
    "f1_cnn": round(cnn_metrics['f1'], 3),
    "f1_hybrid": round(hybrid_metrics['f1'], 3),
    "improvement": round(hybrid_metrics['f1'] - cnn_metrics['f1'], 3),
}

import json
with open(ARTIFACTS_DIR / "metrics" / "hybrid_results.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Summary saved to artifacts/metrics/hybrid_results.json")
print(json.dumps(summary, indent=2))

## 7. Save Results to Git

⚠️ **Запустить после выполнения!** Сохраняет метрики и графики в репозиторий.

In [ ]:
%%bash
cd /content/aie-group-2-sapar
git config user.email "183649607+Sapar-hub@users.noreply.github.com"
git config user.name "Saparmyrat"
git add project/artifacts/metrics/ project/artifacts/figures/
git commit -m "exp06: Hybrid results and metrics"
git push origin main